In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
left = pd.read_csv('data/MLB2024BatterStatsLeft.csv')
left['handed'] = 'L'
right = pd.read_csv('data/MLB2024BatterStatsRight.csv')
right['handed'] = 'R'
raw = pd.read_csv("data/MLB2024BatterStatsRAW.csv")

dirty = pd.concat([left, right])

# Find switch hitters (players with both L and R)
handed_counts = dirty.groupby('player_name')['handed'].nunique()
switch_hitters = handed_counts[handed_counts == 2].index
# Update handedness of switch hitters to 'S'
dirty.loc[dirty.player_name.isin(switch_hitters), 'handed'] = 'S'
handed_df = dirty[['player_id', 'handed']]


# Determine true handedness per player using player_id
true_handed = handed_df.groupby('player_id')['handed'].agg(
    lambda x: 'S' if set(x) == {'L', 'R'} else x.iloc[0]
).reset_index()

# Merge with combined stats
final = raw.merge(true_handed, on='player_id', how='left')

# Add Player name column
final["player_name"] = final["last_name, first_name"]

final = final.sort_values(by='pa', ascending=False)

#excluding players who have less than 2 PA per game AND have played in less than half of the games
final = final[~((final.pa / final.b_game < 2) & (final.b_game < 81))]

final.to_csv('data/MLB2024BatterStats.csv', index=False)
